In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pip install -q pydantic==2.12.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 17.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [3]:
!pip install -q rwkv==0.8.31

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.9/410.9 kB 11.0 MB/s eta 0:00:0000:01


In [25]:
!pip install -qi https://test.pypi.org/simple/ rwkvx==0.2.0.dev8

In [6]:
!pip install -q rwkv huggingface_hub

In [7]:
import os

import huggingface_hub

In [8]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

In [27]:
from rwkvx.generation import (
    RWKVSession,
    RWKVTokenizer,
    RWKVModel,
    RWKVSampler,
    GenerationConfig
)

from rwkvx.conversation import (
    History,
    ConversationManager,
    Message,
    RWKV7PromptRenderer,
    get_default_rwkv7_prompt_templates,
)

In [10]:
model_title = "rwkv7-g1b-2.9b-20251205-ctx8192"
# model_title = "rwkv7-g0a4-7.2b-20251208-ctx8192"
# model_title = "rwkv7-g0b-13.3b-20251130-ctx8192"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")

rwkv7-g1b-2.9b-20251205-ctx8192.pth:   0%|          | 0.00/5.90G [00:00<?, ?B/s]

In [61]:
config = GenerationConfig(
    max_new_tokens=1000,
    stop_tokens=[0, 261],  # 0, \n\n
)

print(config)

GenerationConfig(max_new_tokens=1000, temperature=1.0, top_p=0.3, top_k=0, presence_penalty=0.5, frequency_penalty=0.5, decay_penalty=0.996, stop_tokens=[0, 261])


In [12]:
model = RWKVModel(model_path=model_path)

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/fac6e8b05751a73f51422536be922156d1b5b06c/rwkv7-g1b-2.9b-20251205-ctx8192 (cuda fp16)



In [62]:
tokenizer = RWKVTokenizer(model)
sampler = RWKVSampler(config)

In [63]:
templates = get_default_rwkv7_prompt_templates()
renderer = RWKV7PromptRenderer(templates)

mgr = ConversationManager(renderer)

In [64]:
session = RWKVSession(model, tokenizer, sampler, mgr, config)

In [65]:
session.reset()

In [66]:
mgr.add_system('Include emojis to all your answers.')
mgr.build_prompt()

'System: Include emojis to all your answers.'

In [67]:
mgr.add_user('Tell me a short joke.')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think'

In [68]:
joke2 = session.generate_message()
print(joke2)

role='assistant' text='\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [69]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾'

In [70]:
mgr.add_user('Tell me a joke different from previous one.')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think'

In [71]:
joke = session.generate_message()
print(joke)

role='assistant' text='\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [72]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗'

In [73]:
mgr.add_user('Tell me more complex joke.')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think'

In [74]:
joke3 = session.generate_message()
print(joke3)

role='assistant' text='\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [75]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾'

In [76]:
mgr.add_user('List topics of all jokes that you have told me. Explain the second joke.', metadata={'think_mode': ' think'})
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think'

In [77]:
final_answer = session.generate_message()

In [78]:
print(final_answer.metadata['think_text'])

We need to list all jokes we have told so far. The user wants a list of topics and explanation for the second joke. Let's recall:
1. First joke: "Why did the scarecrow win an award?" Explanation: It was outstanding in its field (corn). So topic: corn, agriculture, awards.
2. Second joke: "Why did the tomato turn red?" Explanation: It saw the salad dressing (tomato sauce). So topic: tomatoes, food, color change.
We need to present a list of topics and explain each joke with that explanation. Also maybe include emojis as per previous style. Provide final answer.


In [79]:
print(final_answer.text)

**List of Jokes & Their Topics**  
| # | Joke | Topic(s) | Explanation |
|---|------|----------|-------------|
| 1 | “Why did the scarecrow win an award?” | Corn, Agriculture, Awards | The scarecrow was outstanding in its field (corn), so it received an award for excellence in farming. |
| 2 | “Why did the tomato turn red?” | Tomatoes, Food, Color Change | The tomato saw the salad dressing (tomato sauce) and turned red because it was excited to be part of a delicious dish! |


In [80]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think>We need to list all jokes we have told so far. The user wants a list of topics and explanation for the second joke. Let\'s recall:\n1. First joke: "Why did the scarecrow win an award?" Explanation: It was outstanding in its field (corn). So topic: corn, agriculture, awards.\n2. Second joke: "Why did the tomato turn red?" Explanation: It saw the salad dressing (tomato sauce). So topic: tom

In [81]:
mgr.add_user('Lets wrap up. Thank you!')
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Why did the tomato turn red? 🍅 Because it saw the salad dressing! 🥗\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Why did the scarecrow win an award? 🏆 Because he was outstanding in his field! 🌾\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think>We need to list all jokes we have told so far. The user wants a list of topics and explanation for the second joke. Let\'s recall:\n1. First joke: "Why did the scarecrow win an award?" Explanation: It was outstanding in its field (corn). So topic: corn, agriculture, awards.\n2. Second joke: "Why did the tomato turn red?" Explanation: It saw the salad dressing (tomato sauce). So topic: tom

In [82]:
session.generate_message()

Message(role='assistant', text="\nYou're welcome! 😊 If you ever want more jokes or need help with anything else, just let me know! Have a great day! 🌟", metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'})